In [5]:
import json

with open('../json/adds_w_vio_ids.json', 'r') as f:
    addresses = json.load(f)

with open('../json/vios.json', 'r') as f:
    vios = json.load(f)

# peek correctly
list(addresses.keys())          # ['BROOKLYN', 'MANHATTAN', ...]
addresses['BROOKLYN'][0]        # first record

{'Full_Address': '1 LORRAINE STREET BROOKLYN 11231',
 'NYCHA_ViolationIDs': [18220152,
  18220153,
  18220154,
  18220149,
  18220150,
  18220151,
  18348421,
  18348422,
  18348423,
  18348424,
  18348425,
  18348426,
  18622391,
  18622392],
 'HPD_ViolationIDs': [5655038,
  5655039,
  5655040,
  5655041,
  8119910,
  8119914,
  8119915]}

In [9]:
# Build NYCHA coords lookup too
nycha_vio_to_coords = {}
for record in vios['nycha_vios']:
    vid = record['Violation ID']  # note the space, based on your screenshot
    if record.get('Latitude') is not None:
        nycha_vio_to_coords[vid] = {
            'Latitude': record['Latitude'],
            'Longitude': record['Longitude']
        }

enriched = {}

for borough, records in addresses.items():
    enriched_records = []
    for record in records:
        lat, lng = None, None
        
        # Try HPD first
        hpd_ids = record.get('HPD_ViolationIDs')
        for vid in (hpd_ids if isinstance(hpd_ids, list) else []):
            match = vio_to_coords.get(vid)
            if match:
                lat = match['Latitude']
                lng = match['Longitude']
                break
        
        # Fall back to NYCHA if still no coords
        if lat is None:
            nycha_ids = record.get('NYCHA_ViolationIDs')
            for vid in (nycha_ids if isinstance(nycha_ids, list) else []):
                match = nycha_vio_to_coords.get(vid)
                if match:
                    lat = match['Latitude']
                    lng = match['Longitude']
                    break

        enriched_records.append({**record, 'Latitude': lat, 'Longitude': lng})
    enriched[borough] = enriched_records

total = sum(len(v) for v in enriched.values())
with_coords = sum(
    1 for records in enriched.values()
    for r in records if r['Latitude'] is not None
)
print(f"Total: {total} | With coords: {with_coords} | Missing: {total - with_coords}")

Total: 227 | With coords: 227 | Missing: 0


In [10]:
with open('../json/addresses_with_coords.json', 'w') as f:
    json.dump(enriched, f)

In [1]:
import json

with open('../json/adds_w_vio_ids.json', 'r') as f:
    raw = f.read()

# Search for NaN as a bare value vs null
print('"NaN"' in raw)   # True = bad, it's a string
print('null' in raw)    # True = good, proper JSON null

False
False
